# Trois méthodes d'entraînement pour un Vision Transformer

| | |
|---|---|
| **Papiers** | ViT [Dosovitskiy et al. (2020)](https://arxiv.org/abs/2010.11929), MAE [He et al. (2021)](https://arxiv.org/abs/2111.06377), DINO [Caron et al. (2021)](https://arxiv.org/abs/2104.14294) |
| **Architecture** | Vision Transformer (ViT) |
| **Catégorie** | Classification supervisée · Reconstruction auto-supervisée · Distillation auto-supervisée |
| **Thème** | Un même backbone, trois signaux d'apprentissage différents |

---

Ce notebook explore une question centrale de l'apprentissage de représentations visuelles : **comment entraîner un Vision Transformer ?**

La réponse dépend de ce qu'on possède et de ce qu'on cherche à apprendre. Si on dispose d'étiquettes et que l'on cherche à apprendre une tâche de classification, la classification supervisée est directe et puissante. Si les images sont disponibles sans annotation (ce qui est souvent le cas à grande échelle) deux familles auto-supervisées s'imposent : le Masked Autoencoder (MAE), qui apprend par reconstruction i.e a reconstituer des patch manquants dans une image, et DINO, qui apprend par distillation entre vues, pour apprendre des représentations invariantes pouvant etre ensuite utilisées dans differentes tâches.

Les trois méthodes partagent le même encodeur ViT. Ce qui change d'une approche à l'autre, c'est le signal d'apprentissage (la fonction de perte) et la tête qui s'y connecte. Même architecture, mais trois conceptions différentes de ce qu'est une *bonne représentation*.

## Imports et configuration commune

In [ ]:
import copy
import math

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from vit_from_scratch import (
    ViTConfig,
    VisionTransformer,
    build_classification_model,
    classification_loss,
    classification_train_step,
    MaskedAutoencoder,
    patchify,
    unpatchify,
    random_patch_mask,
    masked_reconstruction_loss,
    mae_train_step,
    dino_loss,
    dino_train_step,
    update_teacher_ema,
    DINOCenter,
    build_dataloaders,
    CIFAR10_CLASSES,
    CIFAR10_MEAN,
    CIFAR10_STD,
    unnormalize_image,
    plot_training_curves,
    plot_predictions,
    plot_class_attention,
    set_seed,
    get_device,
    train,
)

set_seed(42)
DEVICE = get_device()
print(f"Device : {DEVICE}")

# Données CIFAR-10 (sous-ensemble pour exécution raisonnable)
dl = build_dataloaders(
    "cifar10",
    data_dir="../data",
    batch_size=128,
    max_train_samples=5000,
    max_val_samples=1000,
    download=True,
    image_size=32,
)
print(f"Train : {len(dl.train_loader.dataset)} images")
print(f"Val   : {len(dl.val_loader.dataset)} images")
print(f"Classes : {dl.class_names}")

# Configuration partagée pour les trois méthodes
CONFIG = ViTConfig(
    image_size=32,
    patch_size=4,
    in_channels=3,
    num_classes=10,
    embed_dim=128,
    depth=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.0,
    attention_dropout=0.0,
    position_embedding="learned",
)
NUM_PATCHES = (CONFIG.image_size // CONFIG.patch_size) ** 2
PATCH_DIM = CONFIG.in_channels * CONFIG.patch_size ** 2
print(f"\nArchitecture : {CONFIG.depth} blocs, dim={CONFIG.embed_dim}, têtes={CONFIG.num_heads}")
print(f"Patches : {NUM_PATCHES} de {CONFIG.patch_size}x{CONFIG.patch_size} (dim {PATCH_DIM})")

In [ ]:
# Aperçu du jeu de données
sample_images, sample_labels = next(iter(dl.train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx, ax in enumerate(axes.flat):
    img = unnormalize_image(sample_images[idx], CIFAR10_MEAN, CIFAR10_STD)
    ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
    ax.set_title(dl.class_names[sample_labels[idx]], fontsize=11)
    ax.axis("off")
plt.suptitle("Echantillon CIFAR-10", fontsize=13)
plt.tight_layout()
plt.show()

## Un backbone, trois objectifs

### Le Vision Transformer en bref

Avant de comparer les méthodes d'entraînement, rappelons brièvement ce que fait le ViT : il découpe l'image en petites tuiles carrées (les *patches*), projette chacune dans un espace vectoriel, ajoute un token spécial `[CLS]` et des embeddings de position, puis fait passer le tout dans une pile de blocs Transformer.

$$
\underbrace{\text{image}}_{H \times W \times C}
\;\xrightarrow{\text{découpage}}\;
\underbrace{[x_1, \dots, x_N]}_{N \text{ patches}}
\;\xrightarrow{\text{projection}}\;
\underbrace{[z_0, z_1, \dots, z_N]}_{\text{embeddings} + \text{CLS}}
\;\xrightarrow{\text{encodeur}}\;
\underbrace{h}_{\text{représentation}}
$$

La sortie $h$ (typiquement le vecteur `[CLS]`) est la représentation globale de l'image. **C'est exactement ce vecteur que les trois méthodes d'entraînement vont façonner différemment.**

---

### Ce qui varie d'une méthode à l'autre

| Méthode | Ce qu'on connecte au backbone | Signal d'apprentissage | Étiquettes requises |
|---|---|---|---|
| Classification supervisée | Tête linéaire $K$ classes | Cross-entropie avec étiquettes | ✅ Oui |
| Masked Autoencoder | Décodeur léger de reconstruction | MSE sur patches masqués | ❌ Non |
| DINO | Tête de projection shared avec teacher | Divergence de distributions entre vues | ❌ Non |

L'encodeur ViT est identique dans les trois cas. Ce sont la tête et la perte qui définissent **ce que le modèle apprend à représenter**.

---
## 1. Classification supervisée

### Intuition et motivation

La classification supervisée est l'approche la plus directe : on montre au modèle une image accompagnée de son étiquette (« chat », « voiture »…), et on lui demande de prédire cette étiquette. Le signal est fort et ciblé : on sait exactement ce qu'on veut que le modèle distingue.

Imaginons un bibliothécaire qui apprend à ranger des livres en regardant leur couverture. Un superviseur lui dit à chaque fois « science-fiction », « policier », « roman »... Après assez d'exemples, le bibliothécaire développe une *intuition* des éléments visuels qui distinguent les genres. C'est exactement ce que fait la classification supervisée : le modèle apprend des features discriminatives guidées par les catégories fournies.

### La perte de classification

L'encodeur ViT produit un vecteur de représentation $h \in \mathbb{R}^d$ (le token CLS). Une tête linéaire le projette en $K$ logits, puis softmax donne une distribution de probabilité sur les classes :

$$
p = \mathrm{softmax}(W h + b), \quad p_k = \frac{e^{W_k h}}{\sum_{j=1}^K e^{W_j h}}
$$

La perte de cross-entropie mesure la divergence entre la distribution prédite $p$ et la vérité terrain $y$ (one-hot) :

$$
\mathcal{L}_{\text{cls}} = -\sum_{k=1}^{K} y_k \log p_k
$$

Elle est minimale quand le modèle assigne toute sa masse à la bonne classe ($p_k = 1$ pour la vraie classe). En pratique, on la calcule directement sur les logits (avant softmax) pour des raisons de stabilité numérique.

### Pipeline

$$
x \;\xrightarrow{\text{patches}}\; [z_0 \cdots z_N] \;\xrightarrow{\text{encodeur}}\; h_{\text{CLS}} \;\xrightarrow{\text{tête linéaire}}\; \text{logits} \;\xrightarrow{\mathcal{L}_{\text{cls}}}\; \text{gradient}
$$

Seul le token `[CLS]` est utilisé pour la classification. Les tokens de patches enrichissent la représentation globale via l'attention, mais ne sont pas lus directement par la tête.

### Hyperparamètres

| Paramètre | Valeur |
|---|---|
| Epoques | 50 |
| Optimiseur | AdamW |
| Learning rate | 1e-3 |
| Weight decay | 1e-4 |
| Batch size | 128 |

In [ ]:
# --- Entraînement classification supervisée ---
classifier = build_classification_model(CONFIG)
n_params = sum(p.numel() for p in classifier.parameters())
print(f"Paramètres : {n_params:,}")

optimizer = torch.optim.AdamW(classifier.parameters(), lr=1e-3, weight_decay=1e-4)
history_cls = train(
    model=classifier,
    train_loader=dl.train_loader,
    val_loader=dl.val_loader,
    optimizer=optimizer,
    device=DEVICE,
    epochs=50,
)
print(f"\nVal accuracy finale : {history_cls['val_accuracy'][-1]:.1%}")

In [ ]:
plot_training_curves(history_cls)
plt.show()

### Lecture des courbes

Quelques repères pour interpréter les courbes d'entraînement :

- **Train loss qui descend, val loss qui stagne** : le modèle commence a mémoriser les exemples d'entraînement (surapprentissage). Normal avec un petit dataset comme notre sous-ensemble de 5000 images.
- **Gap train/val accuracy** : un écart croissant confirme le surapprentissage. Avec 5000 images et un ViT de plusieurs centaines de milliers de paramètres, c'est attendu.
- **Plateau de val accuracy** : la capacité de généralisation plafonne pour cette taille de données. Plus de données, du dropout ou de l'augmentation permettraient d'aller plus loin.

In [ ]:
# Prédictions sur un batch de validation
classifier.eval()
val_images, val_labels = next(iter(dl.val_loader))
with torch.no_grad():
    val_logits = classifier(val_images.to(DEVICE))

fig = plot_predictions(
    images=val_images[:8],
    labels=val_labels[:8],
    logits=val_logits[:8].cpu(),
    class_names=dl.class_names,
)
plt.show()

In [ ]:
# Cartes d'attention du token CLS
classifier.eval()
with torch.no_grad():
    _, attention_maps = classifier.forward_with_attention(val_images[:4].to(DEVICE))

for i in range(4):
    img = unnormalize_image(val_images[i], CIFAR10_MEAN, CIFAR10_STD)
    attn_i = tuple(a[i:i+1] for a in attention_maps)
    fig = plot_class_attention(attn_i, img, patch_size=CONFIG.patch_size)
    plt.show()

### Ce que le modèle apprend, et ce qu'il ignore

La classification supervisée produit des représentations **discriminatives** : le modèle apprend à séparer les classes fournies. Il capture efficacement les features qui distinguent « chat » de « chien », mais peut ignorer des régularités visuelles qui n'aident pas cette séparation spécifique, textures, structures spatiales, relations entre régions.

Quand l'utiliser : quand on dispose d'étiquettes fiables et qu'on cible une tâche de classification précise. C'est la méthode de référence, la plus simple à stabiliser et à évaluer.

Limite principale : la représentation apprise est *liée aux classes d'entraînement*. Transférer vers une tâche différente (détection, segmentation, autre domaine) peut nécessiter un fine-tuning substantiel, car le modèle n'a pas appris à encoder l'ensemble de la structure visuelle.

---
## 2. Masked Autoencoder (MAE)

### Intuition : l'épreuve des trous

Le Masked Autoencoder repose sur une idée simple : **cacher une partie de l'image et forcer le modèle à la reconstruire**. C'est l'analogue visuel du test de closure (*cloze test*) en NLP, la même technique que BERT utilise avec les mots masqués.

Imaginez qu'on vous montre un puzzle dont 75 % des pièces sont cachées. Pour deviner les pièces manquantes, vous devez comprendre le contexte global : la forme d'un visage, la continuité d'un horizon, la texture d'un tissu. Vous ne pouvez pas vous contenter de copier le voisinage immédiat, la majorité des voisins sont aussi masqués. Vous devez *raisonner à longue portée*. C'est exactement la contrainte que le MAE impose au ViT.

### La perte de reconstruction

On masque aléatoirement un sous-ensemble $\mathcal{M}$ des $N$ patches. L'encodeur ne reçoit que les patches **visibles**, et un décodeur léger tente de reconstruire les patches masqués. La perte est le MSE entre reconstruction et original sur les positions masquées uniquement :

$$
\mathcal{L}_{\text{MAE}} = \frac{1}{|\mathcal{M}|} \sum_{i \in \mathcal{M}} \lVert \hat{x}_i - x_i \rVert_2^2
$$

où $x_i$ est le patch original et $\hat{x}_i$ la reconstruction du décodeur. Les patches visibles ne contribuent pas à la perte, le signal est concentré là où il y a de l'incertitude.

### Pipeline

$$
x
\xrightarrow{\text{patchify}}
[x_1, \dots, x_N]
\xrightarrow{\text{masque}}
\{x_i : i \notin \mathcal{M}\}
\xrightarrow{\text{encodeur ViT}}
h_{\text{visibles}}
\xrightarrow{\text{décodeur}}
\hat{x}_{\mathcal{M}}
\xrightarrow{\mathcal{L}_{\text{MAE}}}
\text{gradient}
$$

L'asymétrie est importante : l'encodeur profond ne traite que les tokens visibles (efficacité computationnelle), et un décodeur léger reconstruit toutes les positions. Après pré-entraînement, on retire le décodeur et on ne garde que l'encodeur.

### Ratio de masque : l'importance du 75 %

Le ratio de masque est l'hyperparamètre le plus sensible du MAE :

- **Ratio faible (≈ 25 %)** : beaucoup de contexte visible, reconstruction facile. Le modèle peut interpoler localement sans raisonner à grande échelle. La perte descend vite, mais la représentation reste superficielle.
- **Ratio élevé (≈ 90 %)** : très peu de contexte, tâche très difficile. Le modèle doit extraire le maximum de chaque token visible. La convergence est lente et instable.
- **75 % (point optimal He et al.)** : assez difficile pour forcer le raisonnement contextuel, assez faisable pour converger proprement. C'est le réglage recommandé dans le papier MAE original.

L'analogie du puzzle reste valide : 25 % de pièces manquantes est trivial, 90 % est décourageant, 75 % oblige à construire un modèle mental de la scène.

In [ ]:
# --- Visualisation du masking MAE sur une image réelle ---
sample_img = val_images[0]
patches = patchify(sample_img.unsqueeze(0), CONFIG.patch_size)  # [1, N, patch_dim]
N_patches = patches.shape[1]

MASK_RATIO = 0.75
mask = random_patch_mask(1, N_patches, MASK_RATIO)  # [1, N] bool, True=masqué

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Image originale
img_show = unnormalize_image(sample_img, CIFAR10_MEAN, CIFAR10_STD)
axes[0].imshow(img_show.permute(1, 2, 0).clamp(0, 1).numpy())
axes[0].set_title("Image originale")
axes[0].axis("off")

# Image avec patches masqués (gris)
masked_patches = patches.clone()
masked_patches[0, mask[0]] = 0.5
masked_img = unpatchify(masked_patches, CONFIG.patch_size, CONFIG.image_size)
axes[1].imshow(masked_img[0].permute(1, 2, 0).clamp(0, 1).numpy())
axes[1].set_title(f"Masqué ({MASK_RATIO:.0%} des patches)")
axes[1].axis("off")

# Patches visibles uniquement
visible_patches = torch.zeros_like(patches)
visible_patches[0, ~mask[0]] = patches[0, ~mask[0]]
visible_img = unpatchify(visible_patches, CONFIG.patch_size, CONFIG.image_size)
axes[2].imshow(visible_img[0].permute(1, 2, 0).clamp(0, 1).numpy())
axes[2].set_title(f"Patches visibles ({1 - MASK_RATIO:.0%})")
axes[2].axis("off")

plt.suptitle("Masked Autoencoder : masking sur une image CIFAR-10", fontsize=13)
plt.tight_layout()
plt.show()
print(f"Total : {N_patches} patches, masqués : {mask.sum().item()}, visibles : {(~mask).sum().item()}")

### Hyperparamètres MAE

| Paramètre | Valeur |
|---|---|
| Epoques | 50 |
| Mask ratio | 0.75 |
| Learning rate | 1.5e-3 |
| Weight decay | 5e-2 |
| Decoder embed dim | 64 |
| Decoder depth | 1 |

In [ ]:
# --- Entraînement MAE ---
mae_model = MaskedAutoencoder(CONFIG)
n_params_mae = sum(p.numel() for p in mae_model.parameters())
print(f"Paramètres MAE : {n_params_mae:,}")

mae_optimizer = torch.optim.AdamW(mae_model.parameters(), lr=1.5e-3, weight_decay=0.05)
mae_model.to(DEVICE)

mae_losses = []
MAE_EPOCHS = 50

for epoch in range(MAE_EPOCHS):
    mae_model.train()
    epoch_loss = 0.0
    n_batches = 0
    for batch in dl.train_loader:
        images = batch[0].to(DEVICE) if isinstance(batch, (list, tuple)) else batch.to(DEVICE)
        metrics = mae_train_step(
            model=mae_model,
            images_or_batch=images,
            optimizer=mae_optimizer,
            device=DEVICE,
            mask_ratio=MASK_RATIO,
        )
        epoch_loss += metrics["loss"]
        n_batches += 1
    mae_losses.append(epoch_loss / n_batches)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:3d}/{MAE_EPOCHS} - loss : {mae_losses[-1]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(mae_losses) + 1), mae_losses, color="tab:blue")
ax.set_xlabel("Epoque")
ax.set_ylabel("Loss de reconstruction")
ax.set_title("MAE : courbe de perte")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Lecture de la courbe MAE

- **Loss haute au départ** : le modèle prédit approximativement la moyenne des pixels, sans structure.
- **Descente rapide** dans les premières époques : le décodeur apprend les statistiques globales (couleur moyenne, contrastes).
- **Plateau** : le modèle reconstruit les textures basses fréquences mais bute sur les détails fins. Avec 5000 images et 50 époques, c'est un résultat normal.
- Ce qui compte n'est pas la qualité pixel de la reconstruction, mais la richesse des représentations que l'encodeur développe pour permettre cette reconstruction.

In [ ]:
# --- Visualisation des reconstructions MAE ---
mae_model.eval()
vis_images = val_images[:4].to(DEVICE)

with torch.no_grad():
    target_patches = patchify(vis_images, CONFIG.patch_size)
    N_vis = target_patches.shape[1]
    patch_mask = random_patch_mask(4, N_vis, MASK_RATIO, device=DEVICE)
    pred_patches = mae_model(vis_images, patch_mask)  # [B, N, patch_dim]

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
col_titles = ["Original", "Masqué", "Reconstruction", "Erreur"]

for i in range(4):
    img_orig = unnormalize_image(vis_images[i].cpu(), CIFAR10_MEAN, CIFAR10_STD)
    axes[i, 0].imshow(img_orig.permute(1, 2, 0).clamp(0, 1).numpy())

    # Masqué (gris sur les patches masqués)
    masked_p = target_patches[i].clone().cpu()
    masked_p[patch_mask[i].cpu()] = 0.5
    masked_img = unpatchify(masked_p.unsqueeze(0), CONFIG.patch_size, CONFIG.image_size)[0]
    axes[i, 1].imshow(masked_img.permute(1, 2, 0).clamp(0, 1).numpy())

    # Reconstruction (patches masqués remplacés par la prédiction)
    recon_p = target_patches[i].clone().cpu()
    recon_p[patch_mask[i].cpu()] = pred_patches[i][patch_mask[i]].cpu()
    recon_img = unpatchify(recon_p.unsqueeze(0), CONFIG.patch_size, CONFIG.image_size)[0]
    axes[i, 2].imshow(recon_img.permute(1, 2, 0).clamp(0, 1).numpy())

    # Erreur pixel
    error = (img_orig - recon_img.clamp(0, 1)).abs().mean(dim=0)
    axes[i, 3].imshow(error.numpy(), cmap="hot", vmin=0, vmax=0.5)

for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=12)
for ax in axes.flat:
    ax.axis("off")

plt.suptitle("MAE : reconstruction des patches masqués", fontsize=14)
plt.tight_layout()
plt.show()

### Ce que le modèle apprend

L'entraînement MAE pousse l'encodeur à produire des représentations **contextuelles et complètes** : pour reconstruire un patch manquant, chaque token visible doit capter les dépendances locales (texture, bord) *et* globales (forme, objet).

Contrairement à la classification supervisée, le MAE est **génératif** dans son signal : il doit modéliser la distribution des pixels, pas seulement séparer des catégories. Cela tend à produire des représentations plus riches, utiles pour le transfert vers des tâches structurelles (segmentation, détection) où la géométrie locale importe.

Quand l'utiliser : quand on a beaucoup d'images sans étiquettes et qu'on veut pré-entraîner un encodeur polyvalent. L'encodeur peut ensuite être *fine-tuné* avec peu d'étiquettes sur la tâche cible.

---
## 3. DINO : distillation auto-supervisée teacher/student

### Intuition : deux regards sur la même scène

DINO repose sur une idée très différente du MAE. Plutôt que de reconstruire des pixels, l'objectif est le suivant : **deux vues différentes de la même image doivent produire la même représentation, ainsi que s'affranchir du monde des pixel pour raisonner de manière plus abstraite**.

Imaginez deux photographes qui prennent des clichés du même bâtiment, l'un en plan large, l'autre en gros plan sur l'entrée. Une personne qui comprend *vraiment* l'architecture devrait reconnaître que c'est le même bâtiment. DINO impose exactement cela au modèle : peu importe le recadrage, l'échelle ou la luminosité, la représentation doit rester cohérente.

Ce qui rend DINO élégant, c'est qu'il n'a pas besoin d'étiquettes pour définir cette cohérence. Les deux « regards » sont générés automatiquement par des augmentations aléatoires, pas de supervision humaine requise.

### Le duo student/teacher

DINO utilise **deux réseaux** de même architecture :
- Le **student** reçoit des vues diverses (locales et globales) et est entraîné par rétropropagation.
- Le **teacher** reçoit uniquement des vues globales et produit les *cibles* pour le student. Ses poids **ne sont pas mis à jour par rétropropagation**, ils suivent le student via une moyenne mobile exponentielle.

Le student doit prédire la distribution du teacher, ce qui le force à développer une représentation aussi riche et stable que lui, tout en recevant des vues potentiellement plus partielles.

### Les distributions et la perte

Chaque réseau produit des logits $z$, convertis en distribution de probabilité par softmax avec une température $\tau$ :

$$
p_t = \mathrm{softmax}\!\left(\frac{z_t - c}{\tau_t}\right), \qquad
p_s = \mathrm{softmax}\!\left(\frac{z_s}{\tau_s}\right)
$$

- $\tau_t$ (teacher) est petite ($\approx 0.04$) : distribution *piquée*, cibles contrastées.
- $\tau_s$ (student) est plus grande ($\approx 0.1$) : distribution plus *lisse*, laisse au student une certaine tolérance.
- $c$ est un vecteur de **centrage** soustrait des logits du teacher pour éviter l'effondrement vers une distribution uniforme.

La perte est une cross-entropie entre la cible teacher et la prédiction student :

$$
\mathcal{L}_{\text{DINO}} = -\sum_k p_t^{(k)} \log p_s^{(k)}
$$

### Multi-crop : local-to-global correspondence

DINO extrait plusieurs **vues** de l'image à chaque itération :
- **Vues globales** (recadrage large, $\geq 50\%$ de l'image) : passent dans le teacher *et* le student.
- **Vues locales** (petits recadrages, $5$–$40\%$) : passent *uniquement* dans le student.

L'objectif est que chaque vue locale du student corresponde à la vue globale du teacher, **correspondance local-to-global**. Le student doit inférer la représentation globale depuis un fragment partiel. C'est une contrainte encore plus forte que de faire correspondre deux vues globales.

$$
\mathcal{L}_{\text{total}} = \sum_{v \in \text{vues student}} \sum_{g \in \text{vues teacher}} \mathcal{L}_{\text{DINO}}(p_t^{(g)},\, p_s^{(v)})
$$

### Mise à jour du teacher : EMA

Le teacher n'est **jamais entraîné directement**. Ses paramètres $\theta_t$ sont une moyenne mobile exponentielle des paramètres du student $\theta_s$ :

$$
\theta_t \leftarrow m\,\theta_t + (1 - m)\,\theta_s, \qquad m \in [0.99, 0.999]
$$

Cette mise à jour produit un teacher qui est une version **lissée et légèrement retardée** du student. Trois effets clés :
1. **Stabilité** : le teacher évolue lentement, ce qui stabilise les cibles et évite les oscillations.
2. **Pas d'effondrement** : si on entraînait le teacher directement, le système pourrait converger trivialement vers une solution constante. L'EMA brise cette symétrie.
3. **Accumulation** : sur $T$ étapes, le teacher *intègre* l'historique récent du student, une forme de consensus temporel.

In [ ]:
# --- Visualisation des vues multi-crop DINO ---
from vit_from_scratch.dino import build_dino_views, DINOViewConfig

dino_img = val_images[:1].to(DEVICE)  # [1, C, H, W]

view_config = DINOViewConfig(
    teacher_global_crops=2,
    student_global_crops=2,
    student_local_crops=4,
    teacher_global_crop_scale=(0.6, 1.0),
    student_global_crop_scale=(0.6, 1.0),
    student_local_crop_scale=(0.2, 0.5),
)
views = build_dino_views(dino_img, DEVICE, view_config)

n_teacher = len(views.teacher_views)
n_student = len(views.student_views)
total = 1 + n_teacher + n_student

fig, axes = plt.subplots(1, total, figsize=(3 * total, 3.5))

# Image originale
img_show = unnormalize_image(val_images[0], CIFAR10_MEAN, CIFAR10_STD)
axes[0].imshow(img_show.permute(1, 2, 0).clamp(0, 1).numpy())
axes[0].set_title("Original", fontsize=10)
axes[0].axis("off")

# Vues teacher (globales)
for j, tv in enumerate(views.teacher_views):
    img_t = tv[0].cpu()
    # Les vues sont déjà normalisées, on les affiche telles quelles
    img_t = img_t.permute(1, 2, 0).clamp(0, 1).numpy()
    axes[1 + j].imshow(img_t)
    axes[1 + j].set_title(f"Teacher {j+1}\n(global)", fontsize=9)
    axes[1 + j].axis("off")

# Vues student
offset = 1 + n_teacher
for j, sv in enumerate(views.student_views):
    img_s = sv[0].cpu()
    img_s = img_s.permute(1, 2, 0).clamp(0, 1).numpy()
    axes[offset + j].imshow(img_s)
    label = "global" if j < view_config.student_global_crops else "local"
    axes[offset + j].set_title(f"Student {j+1}\n({label})", fontsize=9)
    axes[offset + j].axis("off")

plt.suptitle("DINO : vues multi-crop depuis une seule image", fontsize=12)
plt.tight_layout()
plt.show()
print(f"Teacher : {n_teacher} vues globales")
print(f"Student : {n_student} vues ({view_config.student_global_crops} globales + {view_config.student_local_crops} locales)")

### Hyperparamètres DINO

| Paramètre | Valeur |
|---|---|
| Epoques | 50 |
| Température teacher ($\tau_t$) | 0.04 |
| Température student ($\tau_s$) | 0.1 |
| Momentum EMA | 0.996 |
| Learning rate | 1e-3 |
| Weight decay | 1e-4 |

In [ ]:
# --- Entraînement DINO ---
student = build_classification_model(CONFIG)
teacher = build_classification_model(CONFIG)
teacher.load_state_dict(student.state_dict())
for p in teacher.parameters():
    p.requires_grad = False

student.to(DEVICE)
teacher.to(DEVICE)

center = DINOCenter(dim=CONFIG.num_classes, momentum=0.9, device=DEVICE)
dino_optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3, weight_decay=1e-4)

DINO_EPOCHS = 50
dino_losses = []
dino_entropies = []

n_params_dino = sum(p.numel() for p in student.parameters())
print(f"Paramètres student : {n_params_dino:,}")

for epoch in range(DINO_EPOCHS):
    student.train()
    teacher.eval()
    epoch_loss = 0.0
    epoch_entropy = 0.0
    n_batches = 0

    for batch in dl.train_loader:
        images = batch[0].to(DEVICE) if isinstance(batch, (list, tuple)) else batch.to(DEVICE)
        metrics = dino_train_step(
            student=student,
            teacher=teacher,
            batch_or_views=images,
            optimizer=dino_optimizer,
            device=DEVICE,
            center=center,
            student_temperature=0.1,
            teacher_temperature=0.04,
            momentum=0.996,
        )
        epoch_loss += metrics["loss"]
        epoch_entropy += metrics.get("teacher_entropy", 0.0)
        n_batches += 1

    dino_losses.append(epoch_loss / n_batches)
    dino_entropies.append(epoch_entropy / n_batches)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:3d}/{DINO_EPOCHS} - loss : {dino_losses[-1]:.4f}, entropy : {dino_entropies[-1]:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, len(dino_losses) + 1), dino_losses, color="tab:blue")
ax1.set_xlabel("Epoque")
ax1.set_ylabel("Loss DINO")
ax1.set_title("Perte DINO")
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, len(dino_entropies) + 1), dino_entropies, color="tab:orange")
ax2.set_xlabel("Epoque")
ax2.set_ylabel("Entropie teacher")
ax2.set_title("Entropie des prédictions teacher")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Lecture des courbes DINO

- **Loss qui descend** : le student apprend progressivement a imiter les distributions du teacher. La convergence est plus lente que pour la classification car le signal est indirect.
- **Entropie du teacher** : le signal le plus important. Si l'entropie s'effondre vers zéro, le teacher concentre ses prédictions sur une seule classe : c'est le mode d'effondrement (*collapse*). Une entropie stable et positive indique un apprentissage sain.
- Le centering (soustraction de la moyenne mobile des sorties) est le mécanisme principal contre l'effondrement : il empêche une dimension de dominer les autres.

In [ ]:
# Attention maps du teacher DINO
teacher.eval()
vis_dino = val_images[:3].to(DEVICE)

with torch.no_grad():
    _, attn_maps = teacher.forward_with_attention(vis_dino)

for i in range(3):
    img = unnormalize_image(vis_dino[i].cpu(), CIFAR10_MEAN, CIFAR10_STD)
    attn_i = tuple(a[i:i+1] for a in attn_maps)
    fig = plot_class_attention(attn_i, img, patch_size=CONFIG.patch_size)
    plt.show()

### Ce que le modèle apprend : vers la découverte d'objets

DINO apprend des représentations **invariantes aux augmentations**, c'est-à-dire stables face aux changements de recadrage, d'échelle, de couleur. Cette invariance est différente de celle apprise par classification : elle ne dépend d'aucune catégorie prédéfinie.

Un résultat intéressant du papier original est que les **cartes d'attention** du teacher DINO font émerger des régions cohérentes de l'image (souvent les objets principaux) sans aucune annotation de boîtes ou de masques. C'est une forme de *découverte d'objets* implicite, obtenue gratuitement par le processus d'entraînement.

Quand l'utiliser : quand on dispose de grandes quantités d'images non étiquetées et qu'on veut des représentations polyvalentes et transférables, k-NN, clustering, fine-tuning avec peu d'étiquettes. DINO est particulièrement efficace pour les tâches où la structure spatiale et les frontières d'objets importent.

---
## Tableau comparatif des trois méthodes

| Méthode | Signal d'entraînement | Supervision | Ce qu'on apprend | Forces | Limites |
|---|---|---|---|---|---|
| **Classification supervisée** | Cross-entropie avec étiquettes | ✅ Forte (étiquettes requises) | Frontières de décision entre classes | Simple, stable, directement évaluable | Lié aux classes d'entraînement ; transfert limité |
| **Masked Autoencoder (MAE)** | MSE de reconstruction sur patches masqués | ❌ Aucune | Dépendances contextuelles (local + global) | Représentations riches ; scalable sans données étiquetées | Pas d'objectif sémantique explicite ; décodeur inutile après pré-entraînement |
| **DINO** | Divergence de distributions entre vues | ❌ Aucune | Invariance aux augmentations ; structure spatiale | Représentations très transférables ; découverte d'objets implicite | Pipeline complexe (températures, centrage, EMA, multi-crop) ; sensible aux hyperparamètres |

---
## Conclusion et perspectives

### Ce que nous avons vu

Les trois méthodes d'entraînement explorent une même architecture ViT sous des angles très différents :

- La **classification supervisée** optimise directement une tâche cible avec un signal fort et ciblé. Elle est la plus simple à mettre en œuvre et la plus directe à évaluer, mais requiert des étiquettes et produit des représentations liées à la tâche.

- Le **Masked Autoencoder** transforme la reconstruction en signal d'apprentissage. En cachant 75 % des patches, il force le modèle à développer une compréhension contextuelle profonde de la structure visuelle, sans aucune annotation.

- **DINO** pousse plus loin en imposant la cohérence entre vues d'une même image. L'invariance apprise est plus générale et les représentations plus transférables, au prix d'un pipeline plus complexe.

### Comment les approches se complètent

Dans la pratique, ces méthodes ne s'excluent pas, elles se *composent* :

1. **Pré-entraîner** avec MAE ou DINO sur un grand corpus non étiqueté.
2. **Fine-tuner** avec supervision sur la tâche cible (souvent quelques centaines d'exemples suffisent).

Ce paradigme *pré-entraînement + fine-tuning* est aujourd'hui le standard pour les grands modèles visuels.

### Quand utiliser laquelle ?

| Situation | Recommandation |
|---|---|
| Données étiquetées abondantes, tâche fixe | Classification supervisée |
| Grandes images non étiquetées, besoin de transfert structurel | MAE |
| Grand corpus non étiqueté, besoin de représentations polyvalentes | DINO |
| Budget d'annotation limité, domaine varié | DINO ou MAE + fine-tuning supervisé |

### Pour aller plus loin

- **MAE v2 / BEiT v2 / CAE** : variantes qui combinent reconstruction et objectifs sémantiques.
- **DINOv2** (Oquab et al., 2023) : DINO à l'échelle avec données curées, représentations visuelles universelles.
- **I-JEPA / V-JEPA** (LeCun et al., 2023) : prédiction dans l'espace des représentations plutôt que dans l'espace des pixels.
- **Distillation supervisée** : combiner un teacher supervisé et un student plus petit pour compresser la connaissance.

Le backbone ViT est resté identique tout au long de ce notebook. Ce qui a changé, c'est notre définition de *ce qu'est une bonne représentation*, et c'est précisément cette question qui fait la richesse du domaine.